# Social Media Sentiment and Network Influence Analysis

This notebook explores tweet sentiment, a heuristic user-authenticity score, mention-network structure, and PageRank-based influence. It was adapted from the original Google Colab notebook for safer GitHub sharing.

## Data setup

Place the required datasets in the notebook working directory, or update the paths in the data-loading cell. Do not upload `kaggle.json`, API keys, private datasets, or personal data to a public repository.


### Kaggle credentials (local setup only)

If downloading datasets through Kaggle, configure the Kaggle API credentials privately in your environment. Never commit `kaggle.json` or any token to GitHub.


### Dataset downloads

The original notebook downloads these datasets from Kaggle: `noorsaeed/indian-election-sentiment-dataset-tweets`, `adritpal08/dataset-of-indian-politics-tweets-and-reactions`, `anveshrao123/indian-political-tweets`, and `kazanova/sentiment140`. Obtain them according to each dataset license and place the expected CSV files in the working directory.


### Extract datasets

Extract downloaded archives locally before running the data-loading section. Dataset file names and columns may differ; update paths and column names as needed.


### Dependencies

Install the dependencies listed in the repository `requirements.txt` before running the notebook.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

In [ ]:
df_net = pd.read_csv("tweets.csv")
df_pol = pd.read_csv("BJP_tweets.csv")
modi_df  = pd.read_csv("modi_reviews.csv")
rahul_df = pd.read_csv("rahul_reviews.csv")


df_extra = pd.concat([modi_df, rahul_df], ignore_index=True)


df_s140 = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None
)
df_s140 = df_s140[[0,5]]
df_s140.columns = ['sentiment','tweet']


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df_s140['clean'] = df_s140['tweet'].apply(clean_text)
df_net['clean'] = df_net['Tweet'].apply(clean_text)
df_pol['clean'] = df_pol['tweet'].apply(clean_text)
df_extra['clean'] = df_extra['Tweet'].apply(clean_text)


In [ ]:
X = df_s140["clean"]
y = df_s140["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_train_pred = model.predict(X_train_vec)
y_test_pred  = model.predict(X_test_vec)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy  = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)


cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Test Data)")
plt.show()



In [ ]:
def get_sentiment(text):
    vec = tfidf.transform([text])
    return model.predict_proba(vec)[0][1]

In [ ]:
all_tweets = pd.concat([df_net, df_pol, df_extra], ignore_index=True)
all_tweets['clean'] = all_tweets['Tweet'].apply(clean_text)
all_tweets['sentiment_score'] = all_tweets['clean'].apply(get_sentiment)


In [ ]:

all_tweets['tweet_len'] = all_tweets['Tweet'].apply(lambda x: len(str(x)))
all_tweets = all_tweets[all_tweets['tweet_len'] >= 20]


def vocab_richness(text):
    words = text.split()
    return len(set(words)) / (len(words)+1)

all_tweets['vocab'] = all_tweets['clean'].apply(vocab_richness)


auth_df = all_tweets.groupby('User').agg({
    'Tweet':'count',
    'sentiment_score':'std',
    'vocab':'mean'
}).reset_index()
auth_df.columns = ['user','tweet_count','sentiment_var','vocab_score']


auth_df['authenticity'] = (auth_df['vocab_score'] + auth_df['sentiment_var']) / auth_df['tweet_count']


df_net_auth = all_tweets[all_tweets['User'].isin(auth_df['user'])]


In [ ]:
def label_sentiment(score):
    if score > 0.6:
        return 'Positive'
    elif score < 0.4:
        return 'Negative'


df_net_auth['sentiment_label'] = df_net_auth['sentiment_score'].apply(label_sentiment)



df_net_auth.loc[df_net_auth['clean'].str.contains(r'\b(modi|bjp|nda)\b'), 'target_group'] = 'Modi/BJP'
df_net_auth.loc[df_net_auth['clean'].str.contains(r'\b(rahul|gandhi|congress|upa)\b'), 'target_group'] = 'Rahul/Congress'


In [ ]:

political_df = df_net_auth[df_net_auth['sentiment_label'] != 'Neutral']


counts = political_df.groupby(['target_group','sentiment_label']).size().unstack(fill_value=0)
for col in ['Positive','Negative']:
    if col not in counts.columns:
        counts[col] = 0


import numpy as np

labels = counts.index.tolist()
positive = counts['Positive'].values
negative = counts['Negative'].values
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, positive, width, label='Positive', color='green')
plt.bar(x + width/2, negative, width, label='Negative', color='red')
plt.xticks(x, labels)
plt.ylabel("Number of Tweets")
plt.title("Positive vs Negative Tweets for Modi/BJP and Rahul/Congress (Authentic Users)")
plt.legend()
plt.show()


In [ ]:
def extract_mentions(text):
    return re.findall(r'@(\w+)', str(text))

df_net_auth['mentions'] = df_net_auth['Tweet'].apply(extract_mentions)

G = nx.DiGraph()
for _, row in df_net_auth.iterrows():
    user = row['User']
    for m in row['mentions']:
        G.add_edge(user, m)


In [ ]:
pagerank = nx.pagerank(G)


In [ ]:
pr_df = pd.DataFrame({
    "user": pagerank.keys(),
    "pagerank": pagerank.values()
})

final_df = pr_df.merge(auth_df, left_on="user", right_on="user")
final_df["final_score"] = final_df["pagerank"] * final_df["authenticity"]

top10 = final_df.sort_values(
    by="final_score", ascending=False
).head(10)

top10[[
    "user","pagerank","authenticity",
    "final_score","tweet_count"
]]

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(top10['user'], top10['pagerank'])
plt.xticks(rotation=90)
plt.ylabel("PageRank")
plt.title("Top-10 Influential Authentic Users")
plt.show()


In [ ]:
sorted_df = final_df.sort_values(by='pagerank', ascending=False)
sorted_df['cum_percent'] = sorted_df['pagerank'].cumsum() / sorted_df['pagerank'].sum()

plt.figure(figsize=(6,4))
plt.plot(sorted_df['cum_percent'].values)
plt.axhline(0.8, linestyle='--')
plt.xlabel("Users (sorted by PageRank)")
plt.ylabel("Cumulative Influence")
plt.title("Pareto Curve of Influence")
plt.show()


In [ ]:
print("Total tweets:", len(df_net_auth))

print("With sentiment:",
      df_net_auth['sentiment_label'].notna().sum())

print("With target group:",
      df_net_auth['target_group'].notna().sum())

print("Final usable tweets:",
      df_net_auth.dropna(
          subset=['sentiment_label','target_group']
      ).shape[0])
